# FFA-Based Regional Clustering

Clusters USGS streamflow sites by the **shape** of their flood frequency curves using LP3 scale and skew parameters. Uses Ward hierarchical clustering with a geographic k-NN connectivity constraint — same structure as `cooccurrence_clustering_all.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

## Configuration

In [ ]:
DATA_DIR   = Path("/home/ryan/data/flood_hazard")
OUT_DIR    = Path(".")   # save outputs alongside this notebook

K_NEIGHBORS = 7    # k-NN connectivity constraint; increase if graph not fully connected
N_CLUSTERS  = 5   # set after inspecting dendrogram / silhouette plot

## Load and prepare data

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "ffa" / "flood_frequency.parquet")
meta = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")[["site_no", "latitude", "longitude"]]

# Keep only sites with valid LP3 fits
ffa = ffa[ffa.record_ok & ffa.lp3_scale.notna() & ffa.lp3_skew.notna()]

df = ffa[["site_no", "lp3_scale", "lp3_skew"]].merge(meta, on="site_no").reset_index(drop=True)

print(f"{len(df):,} sites after filtering")
df[["lp3_scale", "lp3_skew"]].describe()

## Filter by hydrologic disturbance

Sites with extreme anthropogenic modification produce unusual LP3 parameters that cause them to cluster as isolated outliers. Filter using `HYDRO_DISTURB_INDX` (composite of dam density, water withdrawal, canal coverage, NPDES proximity, road density, and land fragmentation). Sites not present in GAGES-2 are excluded as unscreened.

Threshold guidance from Falcone et al. (2010): Ref sites have a median index of 8 (75th pct = 11); Non-ref sites have a median of 17. A threshold of ≤ 20 retains ~74% of sites while excluding the most heavily modified watersheds.

In [ ]:
GAGES2_DIR        = Path("/home/ryan/data/usgs/GAGES_2/basinchar_and_report_sept_2011/spreadsheets-in-csv-format")
MAX_DISTURB_INDEX = 17  # adjust to taste; Ref site 75th pct = 11, Non-ref median = 17

gages2 = pd.read_csv(GAGES2_DIR / "conterm_bas_classif.txt", encoding="latin1")
gages2["site_no"] = gages2["STAID"].astype(str).str.zfill(8)

df = df.merge(gages2[["site_no", "CLASS", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")

n_before = len(df)
df = df[df["HYDRO_DISTURB_INDX"].notna() & (df["HYDRO_DISTURB_INDX"] <= MAX_DISTURB_INDEX)].reset_index(drop=True)

print(f"Dropped {n_before - len(df):,} sites (no GAGES-2 match or HYDRO_DISTURB_INDX > {MAX_DISTURB_INDEX})")
print(f"Remaining: {len(df):,} sites")
print()
print(df["CLASS"].value_counts().to_string())

## Winsorize and standardize features

Winsorize at p1/p99 before standardizing to prevent extreme-tailed sites (e.g. very flashy TX/OK streams) from dominating the feature space and getting peeled off into singleton clusters.

In [ ]:
WINSOR_LIMITS = (0.02, 0.02)  # clip bottom and top %

df_clipped = df.copy()
for col in ["lp3_scale", "lp3_skew"]:
    lo = df[col].quantile(WINSOR_LIMITS[0])
    hi = df[col].quantile(1 - WINSOR_LIMITS[1])
    df_clipped[col] = df[col].clip(lo, hi)
    print(f"{col}: clipped [{lo:.4f}, {hi:.4f}]  (raw range [{df[col].min():.4f}, {df[col].max():.4f}])")

scaler = StandardScaler()
X      = scaler.fit_transform(df_clipped[["lp3_scale", "lp3_skew"]])
coords = df[["longitude", "latitude"]].values

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for i, col in enumerate(["lp3_scale", "lp3_skew"]):
    axes[0, i].hist(df[col], bins=60, edgecolor="none")
    axes[0, i].set_title(f"{col} (raw)")
    axes[1, i].hist(df_clipped[col], bins=60, edgecolor="none", color="steelblue")
    axes[1, i].set_title(f"{col} (winsorized)")
plt.tight_layout()
plt.show()

## Build geographic connectivity graph

In [ ]:
connectivity = kneighbors_graph(coords, n_neighbors=K_NEIGHBORS, mode="connectivity", include_self=False)
# Symmetrize
connectivity = connectivity + connectivity.T

n_components = np.linalg.matrix_rank(connectivity.toarray())
print(f"Graph built: {connectivity.nnz // 2:,} edges, k={K_NEIGHBORS}")

# Check connectivity via number of connected components
from scipy.sparse.csgraph import connected_components
n_comp, _ = connected_components(connectivity)
if n_comp > 1:
    print(f"WARNING: graph has {n_comp} connected components — consider increasing K_NEIGHBORS")
else:
    print("Graph is fully connected")

In [ ]:
for k_ in range(3, 30):
    g = kneighbors_graph(coords, n_neighbors=k_, mode='connectivity', include_self=False)
    g = 0.5 * (g + g.T)
    n_comp, _ = connected_components(g, directed=False)
    print(f"k={k_:3d}  →  {n_comp} component(s)")
    if n_comp == 1:
        print(f"  ^^^ minimum valid k")
        break

## Fit Ward clustering

Compute the full linkage matrix for dendrogram inspection, then fit `AgglomerativeClustering` with the connectivity constraint for the final labels.

In [ ]:
# Full linkage (no connectivity constraint) — used only for dendrogram / cut-point selection
Z = linkage(X, method="ward")

# Constrained clustering for final labels
model = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    metric="euclidean",
    linkage="ward",
    connectivity=connectivity,
)
df["cluster"] = model.fit_predict(X) + 1  # 1-indexed

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index().to_string())

## Dendrogram

Large vertical gaps between merges indicate natural cut points.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, truncate_mode="lastp", p=40, no_labels=True, color_threshold=0)
ax.set_title("Ward dendrogram (last 40 merges)")
ax.set_xlabel("Site (or cluster)")
ax.set_ylabel("Ward linkage distance")
plt.tight_layout()
plt.show()

## Acceleration plot

Second derivative of merge distances — peak at k suggests the natural number of clusters.

In [ ]:
last    = Z[-40:, 2]          # last 40 merge distances
accel   = np.diff(last, 2)    # second derivative
rev_acc = accel[::-1]
k_vals  = np.arange(2, len(rev_acc) + 2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_vals, rev_acc, marker="o")
ax.axvline(k_vals[np.argmax(rev_acc)], color="red", linestyle="--", label=f"peak k={k_vals[np.argmax(rev_acc)]}")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Acceleration")
ax.set_title("Acceleration of merge distances")
ax.legend()
plt.tight_layout()
plt.show()

## Silhouette scores

In [ ]:
k_range = range(4, 25)
scores  = []

for k in k_range:
    m = AgglomerativeClustering(n_clusters=k, metric="euclidean", linkage="ward", connectivity=connectivity)
    labels = m.fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = list(k_range)[np.argmax(scores)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), scores, marker="o")
ax.axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Mean silhouette score")
ax.set_title("Silhouette scores (constrained Ward)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best k by silhouette: {best_k}  (score={max(scores):.4f})")

## Cluster map

In [ ]:
cmap   = plt.get_cmap("tab20", N_CLUSTERS)
colors = [cmap(c - 1) for c in df["cluster"]]

fig, ax = plt.subplots(figsize=(14, 8))
sc = ax.scatter(
    df["longitude"], df["latitude"],
    c=df["cluster"], cmap=cmap, vmin=0.5, vmax=N_CLUSTERS + 0.5,
    s=20, linewidths=0, alpha=0.8,
)
plt.colorbar(sc, ax=ax, label="Cluster", ticks=range(1, N_CLUSTERS + 1))
ax.set_title(f"FFA regional clusters (k={N_CLUSTERS}, Ward + k-NN constraint)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## Cluster profiles

Mean LP3 scale and skew per cluster — helps interpret what each region represents.

In [ ]:
profile = df.groupby("cluster")[["lp3_scale", "lp3_skew"]].agg(["mean", "std", "count"])
profile.columns = ["_".join(c) for c in profile.columns]
print(profile.to_string())

## Save outputs

In [ ]:
# out_path = OUT_DIR / f"site_regions_ffa_{N_CLUSTERS}.csv"
# df[["site_no", "cluster", "latitude", "longitude", "lp3_scale", "lp3_skew"]].to_csv(out_path, index=False)
# print(f"Saved: {out_path}")